In [1]:
import numpy as np
import cupy as cp
from scipy.spatial import Delaunay
from scipy.stats import qmc
from typing import Literal
from tqdm import tqdm
import time
import sys
import os
from pyproj import CRS
import dash
import dash.dcc as dcc
import dash.html as html
import plotly.graph_objects as go
import plotly.io as pio
from dash import dcc, html, Input, Output, State

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from workflow.compare.griddiff.core.surfacing import interpolate_surface
from workflow.compare.griddiff.core.grid_volume_computer import GridVolumeComputer
from workflow.utils.data_types import wrap_points_to_pc
from workflow.config.ConfigNode import ConfigNode
from workflow.pointcloud import PointCloud as PC
from workflow.compare.griddiff.plotting.plot_heatmap import plotly_heatmap
from workflow.processor.cutting.point_cloud_grid_cutting import PointCloudGridCutter

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
'''整套流程运行完后, 如果yaml文件中有参数被临时修改, 想观察不同参数的效果, 则需要重新执行该单元格'''
yaml_path = '/home/gary/point-cloud-3d/params/CloudCompare.yaml'
cfg = ConfigNode.load_from_yaml(yaml_path)
cfg.validate()

In [3]:
cloud_A_name = 'DJI_202510101155_157_清云湛江方向-K516-550-650-非重复'
cloud_B_name = 'DJI_202510101259_162_清云湛江方向-K516-550-650-6cm-非重复'

cloud_A_file = os.path.join('/home/gary/CloudPointProcessing/20251010实验', cloud_A_name, 'raw', 'las', 'cloth_nodes.txt')
cloud_B_file = os.path.join('/home/gary/CloudPointProcessing/20251010实验', cloud_B_name, 'raw', 'las', 'cloth_nodes.txt')

cloudA_points = np.genfromtxt(cloud_A_file, delimiter=None, dtype=np.float64)
cloudB_points = np.genfromtxt(cloud_B_file, delimiter=None, dtype=np.float64)

print("读入CSF地面布料网格点...")
print("cloudA_points:", cloudA_points.shape)
print("cloudB_points:", cloudB_points.shape)

pcA_ground = wrap_points_to_pc(cloudA_points[:, 0], cloudA_points[:, 1], cloudA_points[:, 2], CRS("EPSG:4547"),  device='GPU')
pcB_ground = wrap_points_to_pc(cloudB_points[:, 0], cloudB_points[:, 1], cloudB_points[:, 2], CRS("EPSG:4547"), device='GPU')

cutterA_ground = PointCloudGridCutter(pcA_ground)
cutterA_ground.cut(cfg.block_size, pcA_ground.crs, use_cache=False)

cutterB_ground = PointCloudGridCutter(pcB_ground)
cutterB_ground.cut(cfg.block_size, pcB_ground.crs, use_cache=False)

common_keys = set(cutterA_ground.grid_indices) & set(cutterB_ground.grid_indices)
print("common_keys:", len(common_keys))


读入CSF地面布料网格点...
cloudA_points: (7262112, 3)
cloudB_points: (7029912, 3)
2026-01-08 10:58:38 - point_cloud_grid_cutting - INFO - 开始执行点云栅格切割
2026-01-08 10:58:38 - point_cloud_grid_cutting - INFO - 开始网格分组（块尺寸 1米）
2026-01-08 10:58:38 - point_cloud_grid_cutting - INFO - 正在处理 18392 个分块


生成分块索引: 100%|██████████| 18392/18392 [00:03<00:00, 5825.60块/s]

2026-01-08 10:58:41 - point_cloud_grid_cutting - INFO - 切割完成，共生成 18392 个分块
2026-01-08 10:58:41 - point_cloud_grid_cutting - INFO - 开始执行点云栅格切割
2026-01-08 10:58:41 - point_cloud_grid_cutting - INFO - 开始网格分组（块尺寸 1米）
2026-01-08 10:58:41 - point_cloud_grid_cutting - INFO - 正在处理 17969 个分块



生成分块索引: 100%|██████████| 17969/17969 [00:03<00:00, 5772.94块/s]

2026-01-08 10:58:45 - point_cloud_grid_cutting - INFO - 切割完成，共生成 17969 个分块
common_keys: 17969


In [4]:
def get_overlap_bounds(pcA_x, pcA_y, pcB_x, pcB_y):

    a_x_min, a_x_max = cp.min(pcA_x), cp.max(pcA_x)
    a_y_min, a_y_max = cp.min(pcA_y), cp.max(pcA_y)

    b_x_min, b_x_max = cp.min(pcB_x), cp.max(pcB_x)
    b_y_min, b_y_max = cp.min(pcB_y), cp.max(pcB_y)

    inter_x_min = max(a_x_min, b_x_min)
    inter_x_max = min(a_x_max, b_x_max)
    inter_y_min = max(a_y_min, b_y_min)
    inter_y_max = min(a_y_max, b_y_max)

    return (float(inter_x_min), float(inter_x_max)), (float(inter_y_min), float(inter_y_max))

In [5]:
class TINVolumeComputer:
    def __init__(self, cloudA_points, cloudB_points, n_samples=2**12, qmc_type='sobol'):
        """
        基于 Delaunay 三角剖分的不规则三角网 (TIN) 体积计算类
        """
        # --------------------- 局部坐标归一化 ---------------------
        # 使用两个点云 XY 的最小值作为共同原点
        all_xy = np.vstack([cloudA_points[:, :2], cloudB_points[:, :2]])
        self.origin_xy = all_xy.min(axis=0)  # shape: (2,)

        # 平移到局部坐标系
        self.xyA_local = cloudA_points[:, :2] - self.origin_xy
        self.xyB_local = cloudB_points[:, :2] - self.origin_xy

        self.zA = cloudA_points[:, 2]
        self.zB = cloudB_points[:, 2] 

        # --------------------- 构建 TIN ---------------------
        '''
        Qhull 在构建初始单纯形时，要求点不能完全共面、共线或退化，否则会报精度错误
        QJ（Joggle input）：会自动在每个输入点上添加极小的随机扰动（扰动量远小于浮点误差，通常 1e-15 量级），使点集略微不共面，从而成功构建 Delaunay 三角网
        Pp（Precision ignore）：即使 Qhull 在计算过程中检测到潜在的精度问题（如圆整误差较大），也不打印警告信息到控制台
        '''
        self.triA = Delaunay(self.xyA_local, qhull_options='QJ Pp')
        self.triB = Delaunay(self.xyB_local, qhull_options='QJ Pp')

        # --------------------- 公共投影重叠区域（原始坐标系） ---------------------
        self.x_min = max(self.xyA_local[:, 0].min(), self.xyB_local[:, 0].min())
        self.x_max = min(self.xyA_local[:, 0].max(), self.xyB_local[:, 0].max())
        self.y_min = max(self.xyA_local[:, 1].min(), self.xyB_local[:, 1].min())
        self.y_max = min(self.xyA_local[:, 1].max(), self.xyB_local[:, 1].max())

        # print("self.xyA_local:{} self.zA:{} self.x_min:{} x_max:{} y_min:{} y_max:{}".format(self.xyA_local, self.zA, self.x_min, self.x_max, self.y_min, self.y_max))
        # --------------------- QMC 采样基础序列 ---------------------
        self.n_samples = n_samples
        self.qmc_base = self._generate_qmc_base(n_samples, qmc_type)
        self.inside_mask = None

    def get_prism_params(self):
        xy_all = np.concatenate([self.xyA_local, self.xyB_local], axis=0)  # (N_A + N_B, 2)
        z_all  = np.concatenate([self.zA, self.zB], axis=0)  # (N_A + N_B,)

        # 最小二乘法拟合平面 z = a*x + b*y + c
        # 构建矩阵 A = [x, y, 1]
        A = np.concatenate([xy_all, np.ones((xy_all.shape[0], 1))], axis=1)  # (N, 3)
        # lstsq 返回最小二乘解 w = [a, b, c]
        w, _, _, _ = np.linalg.lstsq(A, z_all, rcond=None)
        a, b, c = w

        # 计算所有点相对于趋势平面的残差（偏移量 d）
        d_vals = z_all - (a * xy_all[:, 0] + b * xy_all[:, 1] + c)  # (N,)
        d_min, d_max = np.min(d_vals).item(), np.max(d_vals).item()

        return {
            "plane_coeffs": (a, b, c),  # 趋势平面斜率 (a, b)
            "d_top": d_max,  # 上边界最大偏移
            "d_bottom": d_min,  # 下边界最小偏移
        }
    
    def _generate_qmc_base(self, n, q_type:Literal['sobol', 'halton'] = 'sobol'):
        if q_type == 'sobol':
            m = int(np.log2(n))
            return np.asarray(qmc.Sobol(d=3, scramble=True).random_base2(m=m), dtype=np.float32)
        elif q_type == 'halton':
            return np.asarray(qmc.Halton(d=3, scramble=True).random(n=n), dtype=np.float32)
        else:
            raise ValueError(f"Unsupported QMC type: {q_type}")

    def _interp_tin_z(self, tri: Delaunay, query_xy_local: np.ndarray, z: np.ndarray,) -> np.ndarray:
        """
        在指定的 TIN 表面上对查询点 query_xy (N, 2) 进行线性插值，返回对应的 z 值。

        参数:
            tri        : scipy.spatial.Delaunay 对象（已对 points[:, :2] 构建）
            query_xy   : (N, 2) 的 ndarray，要插值的 xy 坐标（蒙特卡洛采样点）
            z     : (M, 1) 的 ndarray，点云高程 [z]

        返回:
            z_interp   : (N,) 的 ndarray，插值得到的 z 值。
                         如果查询点落在 TIN 凸包外，返回 np.nan
        """
        # 1. 查找每个查询点落在哪个三角形（simplex）内
        simplex_indices = tri.find_simplex(query_xy_local)  # shape: (N,)
        valid_mask = simplex_indices != -1  # 在凸包内的点

        # 初始化结果为 nan（落在外部的点保持 nan）
        z_interp = np.full(len(query_xy_local), np.nan, dtype=np.float64)

        if not np.any(valid_mask):
            return z_interp

        # 取出有效的三角形索引和对应顶点
        valid_idx = simplex_indices[valid_mask]  # 有效的 simplex 索引
        vertices_idx = tri.simplices[valid_idx]  # shape: (N_valid, 3)

        # 对应三个顶点的坐标 (x,y,z)
        tri_zs = z[vertices_idx]  # shape: (N_valid, 3, 3)

        # 使用 Delaunay 内置的 transform 矩阵计算重心坐标（高效且数值稳定）
        T = tri.transform[valid_idx]  # shape: (N_valid, 3, 3)
        # T[:, :2, :2] 是仿射变换基，T[:, 2, :2] 是平移（即一个顶点的坐标）

        # 计算相对偏移
        d = query_xy_local[valid_mask] - T[:, 2, :2]  # shape: (N_valid, 2)

        # 前两个重心坐标（barycentric coords for vertex 0 and 1）
        bary = np.einsum('ijk,ik->ij', T[:, :2, :2], d)  # shape: (N_valid, 2)

        # 第三个坐标 = 1 - sum(前两个)
        bary3 = 1.0 - bary.sum(axis=1)

        # 最终插值 z = bary0 * z0 + bary1 * z1 + bary3 * z2
        z_interp[valid_mask] = (bary[:, 0] * tri_zs[:, 0] +
                                bary[:, 1] * tri_zs[:, 1] +
                                bary3 * tri_zs[:, 2])

        return z_interp

    def compute_volume(self) -> float:
        params = self.get_prism_params()
        a, b, c = params["plane_coeffs"]
        d_min, d_max = params["d_bottom"], params["d_top"]

        # 映射 QMC 随机数到当前网格的斜棱柱空间
        self.rand_x = self.x_min + self.qmc_base[:, 0] * (self.x_max - self.x_min)
        self.rand_y = self.y_min + self.qmc_base[:, 1] * (self.y_max - self.y_min)
        # z 映射到斜面约束范围: z = a*x + b*y + d_min + u*(d_max-d_min)

        self.rand_z = (a * self.rand_x + b * self.rand_y + c + d_min + self.qmc_base[:, 2] * (d_max - d_min))

        rand_xy = np.stack([self.rand_x, self.rand_y], axis=1)
        interp_za = self._interp_tin_z(self.triA, rand_xy, self.zA)
        interp_zb = self._interp_tin_z(self.triB, rand_xy, self.zB)

        # 蒙特卡洛思想评估体积
        valid = ~np.isnan(interp_za) & ~np.isnan(interp_zb)
        inside = (interp_za - self.rand_z) * (interp_zb - self.rand_z) <= 0
        self.inside_mask = valid & inside
        hit_count = np.sum(self.inside_mask)
        v_prism = (self.x_max - self.x_min) * (self.y_max - self.y_min) * (d_max - d_min)
        est_volume = v_prism * hit_count / self.n_samples
        
        return est_volume



In [6]:
def volume_dict_to_heatmap(volume_dict):
    x_coords = [key[0] for key in volume_dict.keys()]
    y_coords = [key[1] for key in volume_dict.keys()]
    x_unique = sorted(set(x_coords))
    y_unique = sorted(set(y_coords))
    nx = len(x_unique)
    ny = len(y_unique)

    x_to_i = {x: i for i, x in enumerate(x_unique)}
    y_to_j = {y: j for j, y in enumerate(y_unique)}

    heatmap_data = np.zeros((ny, nx))

    for (x, y), val in volume_dict.items():
        i = x_to_i[x]
        j = y_to_j[y]
        heatmap_data[j, i] = val

    return heatmap_data, x_unique, y_unique

In [7]:
def visualize_volume_computation(V_computer, max_points=5000):

    params = V_computer.get_prism_params()
    a, b, c = params["plane_coeffs"]
    d_top, d_bottom = params["d_top"], params["d_bottom"]
    x_min, x_max = V_computer.x_min, V_computer.x_max
    y_min, y_max = V_computer.y_min, V_computer.y_max

    # 生成规则网格用于绘制平面
    res = 5
    XI = np.linspace(x_min, x_max, res)
    YI = np.linspace(y_min, y_max, res)
    XI, YI = np.meshgrid(XI, YI)

    # 绘制包围棱柱的上下切面
    Z_plane_top = a * XI + b * YI + c + d_top
    Z_plane_bottom = a * XI + b * YI + c + d_bottom

    # 定义四个角点
    corners_xy = np.array([[x_min, y_min], [x_max, y_min], [x_max, y_max], [x_min, y_max]])
    z_top_corners = a * corners_xy[:, 0] + b * corners_xy[:, 1] + c + d_top
    z_bottom_corners = a * corners_xy[:, 0] + b * corners_xy[:, 1] + c + d_bottom

    fig = go.Figure()

    fig.add_trace(go.Mesh3d(
        x=V_computer.xyA_local[:, 0], y=V_computer.xyA_local[:, 1], z=V_computer.zA,
        i=V_computer.triA.simplices[:, 0], j=V_computer.triA.simplices[:, 1], k=V_computer.triA.simplices[:, 2],
        color='wheat', opacity=0.8, name='曲面A', showlegend=True
    ))

    fig.add_trace(go.Mesh3d(
        x=V_computer.xyB_local[:, 0], y=V_computer.xyB_local[:, 1], z=V_computer.zB,
        i=V_computer.triB.simplices[:, 0], j=V_computer.triB.simplices[:, 1], k=V_computer.triB.simplices[:, 2],
        color='steelblue', opacity=0.8, name='曲面B', showlegend=True
    ))
    
    plane_style = dict(
        colorscale=[[0, "#9B9B9B"], [1, '#646464']],
        opacity=0.2, showscale=False,
        lighting=dict(ambient=0.9, diffuse=0.5, roughness=1.0, specular=0.01, fresnel=0.0)
    )

    fig.add_trace(go.Surface(x=XI, y=YI, z=Z_plane_top, **plane_style, name='上趋势面', showlegend=True))
    fig.add_trace(go.Surface(x=XI, y=YI, z=Z_plane_bottom, **plane_style, name='下趋势面', showlegend=True))

    # 绘制包围盒
    for i in range(4):
        next_i = (i + 1) % 4
        wall_x = [corners_xy[i,0], corners_xy[next_i,0], corners_xy[next_i,0], corners_xy[i,0], corners_xy[i,0]]
        wall_y = [corners_xy[i,1], corners_xy[next_i,1], corners_xy[next_i,1], corners_xy[i,1], corners_xy[i,1]]
        wall_z = [z_bottom_corners[i], z_bottom_corners[next_i], z_top_corners[next_i], z_top_corners[i], z_bottom_corners[i]]
        
        fig.add_trace(go.Scatter3d(
            x=wall_x, y=wall_y, z=wall_z, mode='lines', 
            line=dict(color='black', width=3), showlegend=False
        ))

    rx, ry, rz = V_computer.rand_x, V_computer.rand_y, V_computer.rand_z
    inside_idx = np.where(V_computer.inside_mask == 1)[0][:max_points]

    fig.add_trace(go.Scatter3d(
        x=rx[inside_idx], y=ry[inside_idx], z=rz[inside_idx],
        mode='markers',
        marker=dict(size=1.5, color='green'),
        name='落在两曲面间的随机样本'
    ))

    fig.update_layout(
        title="边坡曲面与体积测算",
        scene=dict(
            xaxis_title="X", yaxis_title="Y", zaxis_title="Z", 
            aspectmode='cube'
        ),
        margin=dict(l=0, r=0, b=0, t=50),
        legend=dict(font=dict(size=18))
    )

    return fig

In [8]:
def launch_grid_volume_dashboard(
    heatmap_data, x_unique, y_unique,
    cutterA_ground,
    cutterB_ground,
    page_title='体积差异看板',
    port=8050
):
    heatmap_fig = plotly_heatmap(heatmap_data, '体积差异热力图', x_unique, y_unique)
    
    # --------- Dash App ---------
    app = dash.Dash(__name__)
    app.title = page_title
    app.layout = html.Div([
        dcc.Store(id="grid_id"),

        html.Div([
            html.Div([
                html.Label("网格ID: "),
                dcc.Input(id='grid_id_display', type='text', readOnly=True, style={'width': '6vw', 'textAlign': 'center'})
            ], style={'display': 'flex', 'flexDirection': 'row', 'alignItems': 'center', 'gap': '0.5vw'}),

            html.Div([
                html.Label("估算体积: "),
                dcc.Input(id='volume_info', type='text', readOnly=True, style={'width': '6vw', 'textAlign': 'center'})
            ], style={'display': 'flex', 'flexDirection': 'row', 'alignItems': 'center', 'gap': '0.5vw'})
        ],
        style={
            'display': 'flex',
            'justifyContent': 'center',
            'alignItems': 'center',
            'gap': '3vw',
        }),
 
        html.Div([
            html.Div([dcc.Graph(id='heatmap_fig', figure=heatmap_fig)], style={'flex': '1', 'height': '70vh'}),
            html.Div([dcc.Graph(id='volume_3d')], style={'flex': '1', 'height': '70vh'}),
        ], style={'display': 'flex', "justifyContent": "center", "alignItems": "center", 'gap': '5vw'}),

    ])

    @app.callback(
        [Output("grid_id", "data"),
        Output("heatmap_fig", "figure"),
        Output("grid_id_display", "value")],
        Input("heatmap_fig", "clickData"),
        State("heatmap_fig", "figure"),
        prevent_initial_call=True
    )
    def on_click(clickData, fig_state):
        if not clickData or not clickData.get("points"):
            return dash.no_update
        
        pt = clickData['points'][0]
        i = int(pt['x'])
        j = int(pt['y'])
        grid_id = {"i": int(i), "j": int(j)}
        
        fig = go.Figure(fig_state) if fig_state else go.Figure()

        fig.data = [trace for trace in fig.data if trace.name != 'clicked_grid']

        fig.add_trace(go.Scatter(
            x=[i], y=[j],
            mode='markers',
            marker=dict(size=10, color='white', symbol='circle-open', line=dict(width=2)),
            name='clicked_grid',
            showlegend=False,
            hoverinfo='skip'
        ))

        grid_id_info = f"({grid_id['i']}, {grid_id['j']})"

        return grid_id, fig, grid_id_info
    
    @app.callback(
        [Output("volume_3d", "figure"),
         Output("volume_info", "value")],
         Input("grid_id", "data"),
         prevent_initial_call=True
    )
    def update_volume_fig(grid_id):

        grid_key = (grid_id["i"], grid_id["j"])

        # 临时计算所选择的网格体积差，不选择保存体积实例
        gridA_indices = cutterA_ground.grid_indices[grid_key]
        gridA_points = cp.stack([cutterA_ground.pc.x[gridA_indices], cutterA_ground.pc.y[gridA_indices], cutterA_ground.pc.z[gridA_indices]], axis=1)

        gridB_indices = cutterB_ground.grid_indices[grid_key]
        gridB_points = cp.stack([cutterB_ground.pc.x[gridB_indices], cutterB_ground.pc.y[gridB_indices], cutterB_ground.pc.z[gridB_indices]], axis=1)

        try:
            V_computer = TINVolumeComputer(gridA_points.get(), gridB_points.get())
            v_est = V_computer.compute_volume()
        except Exception as e:
            empty_fig = go.Figure()
            empty_fig.update_layout(title=f"网格{grid_key}计算失败")
            return empty_fig, f"错误: {str(e)}"
        
        fig = visualize_volume_computation(V_computer)
        volume_info = f"{v_est:.3f} m³"

        return fig, volume_info

    app.run(port=port, debug=True)

In [9]:
start_t = time.time()

i = 0
volume_dict = {}
for grid_key in tqdm(common_keys, desc="计算每个网格的体积差"):

    gridA_indices = cutterA_ground.grid_indices[grid_key]
    gridA_points = cp.stack([cutterA_ground.pc.x[gridA_indices], cutterA_ground.pc.y[gridA_indices], cutterA_ground.pc.z[gridA_indices]], axis=1)

    gridB_indices = cutterB_ground.grid_indices[grid_key]
    gridB_points = cp.stack([cutterB_ground.pc.x[gridB_indices], cutterB_ground.pc.y[gridB_indices], cutterB_ground.pc.z[gridB_indices]], axis=1)

    try:
        computer = TINVolumeComputer(gridA_points.get(), gridB_points.get())
        v_est = computer.compute_volume()
        volume_dict[grid_key] = v_est
    except Exception as e:
        raise e
    
    # raise SystemExit(0)

end_t = time.time()

print(f"估算完成！耗时: {end_t - start_t:.4f} 秒")

heatmap_data, x_unique, y_unique = volume_dict_to_heatmap(volume_dict)



计算每个网格的体积差: 100%|██████████| 17969/17969 [02:38<00:00, 113.47it/s]

估算完成！耗时: 158.3599 秒


In [10]:
launch_grid_volume_dashboard(heatmap_data, x_unique, y_unique, cutterA_ground, cutterB_ground )

In [ ]:
import plotly.graph_objects as go
import random

test_las_file = "/home/gary/CloudPointProcessing/云梧高速/DJI_202506240857_181_云梧-K136-274-404-点云/raw/las/cloud_merged.las"
pc = PC(type='GPU')
pc.load_from_las(test_las_file)

cutter = PointCloudGridCutter(pc)
cutter.cut(1.0, pc.crs, use_cache=False)

In [ ]:

demo_grid = random.choice(list(cutter.grid_indices.keys()))
indices = cutter.grid_indices[demo_grid]
demo_pc = cutter.pc

demo_pc.to_local_origin()

x_np = demo_pc.x[indices].get()
y_np = demo_pc.y[indices].get()
z_np = demo_pc.z[indices].get()

print(len(y_np))
print(y_np)

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=x_np, y=y_np, z=z_np,
    mode='markers',
    marker=dict(size=2, color=z_np, colorscale='Turbo')
))

fig.update_layout(
    scene=dict(
        aspectmode='cube',
        aspectratio=dict(x=1, y=1, z=3),
        xaxis=dict(title=f"X"),
        yaxis=dict(title=f"Y"),
        zaxis=dict(title="Z")
    )
)

fig.show()


In [ ]:
import cupy as cp
import numpy as np
from tqdm import tqdm
import time
import plotly.graph_objects as go
from scipy.integrate import dblquad
from workflow.compare.griddiff.core.grid_volume_computer import GridVolumeComputer

def surface_A(x, y):
    # return np.sin(x) * np.cos(y)
    return x**2 - y**2 + 1.0

def surface_B(x, y):
    # return (x**2 + y**2) + 2.0 * np.exp(-(x**2 + y**2)) + 2.0
    return 2.0 - (x**2 + y**2)

def get_theoretical_volume(func_bottom, func_top, x_range, y_range):
    """
    计算两个曲面之间围成的物理体积（自动处理曲面交叉）
    """
    # 使用 np.abs 确保计算的是 |Z_top - Z_bottom| 的物理体积
    # 这样无论函数在何处交叉，积分结果始终为正且代表实际空间大小
    integrand = lambda y, x: np.abs(func_top(x, y) - func_bottom(x, y))
    
    v_exact, error_est = dblquad(
        integrand, 
        x_range[0], x_range[1], 
        lambda x: y_range[0], lambda x: y_range[1]
    )
    
    # 如果积分器提示误差过大，通常是因为曲面在积分域内剧烈震荡
    return v_exact


def visualize_volume_computation(V_computer, XI, YI, ZA, ZB, max_points=5000):

    # 1. 获取参数
    params = V_computer._get_prism_params()
    a, b = params["plane_coeffs"]
    dt, db = params["d_top"], params["d_bottom"]
    (xmin, xmax), (ymin, ymax) = params["x_range"], params["y_range"]
    
    # 定义四个角点用于侧墙
    corners_xy = np.array([[xmin, ymin], [xmax, ymin], [xmax, ymax], [xmin, ymax]])
    z_top_corners = a * corners_xy[:, 0] + b * corners_xy[:, 1] + dt
    z_bottom_corners = a * corners_xy[:, 0] + b * corners_xy[:, 1] + db

    # 初始化唯一画布
    fig = go.Figure()

    # 绘制原始曲面 (Surface A & B)
    fig.add_trace(go.Surface(x=XI, y=YI, z=ZA, colorscale='Blues', opacity=0.95, showscale=False, name='Surface A'))
    fig.add_trace(go.Surface(x=XI, y=YI, z=ZB, colorscale='Reds', opacity=0.95, showscale=False, name='Surface B'))

    # 绘制包围棱柱的上下切面
    Z_plane_top = a * XI + b * YI + dt
    Z_plane_bottom = a * XI + b * YI + db

    # 并通过 lighting 减弱高光
    plane_style = dict(
        colorscale=[[0, 'rgb(34, 139, 34)'], [1, 'rgb(0, 100, 0)']], # 森林绿到深绿
        opacity=0.25,
        showscale=False,
        lighting=dict(ambient=0.9, diffuse=0.5, roughness=1.0, specular=0.01, fresnel=0.0)
    )

    fig.add_trace(go.Surface(x=XI, y=YI, z=Z_plane_top, **plane_style, name='Top Plane'))
    fig.add_trace(go.Surface(x=XI, y=YI, z=Z_plane_bottom, **plane_style, name='Bottom Plane'))

    # 获取并筛选内部点
    rx, ry, rz = V_computer.rand_x.get(), V_computer.rand_y.get(), V_computer.rand_z.get()
    
    inside_idx = np.where(V_computer.inside_mask.get() == 1)[0][:max_points]

    fig.add_trace(go.Scatter3d(
        x=rx[inside_idx], y=ry[inside_idx], z=rz[inside_idx],
        mode='markers',
        marker=dict(size=1.5, color='green'),
        name='体积内随机样本'
    ))

    # 绘制包围盒
    for i in range(4):
        next_i = (i + 1) % 4
        wall_x = [corners_xy[i,0], corners_xy[next_i,0], corners_xy[next_i,0], corners_xy[i,0], corners_xy[i,0]]
        wall_y = [corners_xy[i,1], corners_xy[next_i,1], corners_xy[next_i,1], corners_xy[i,1], corners_xy[i,1]]
        wall_z = [z_bottom_corners[i], z_bottom_corners[next_i], z_top_corners[next_i], z_top_corners[i], z_bottom_corners[i]]
        
        fig.add_trace(go.Scatter3d(
            x=wall_x, y=wall_y, z=wall_z, mode='lines', 
            line=dict(color='black', width=3), showlegend=False
        ))

    # 布局设置
    fig.update_layout(
        title="Refined Enclosing Prism",
        scene=dict(
            xaxis_title="X", yaxis_title="Y", zaxis_title="Z", 
            aspectmode='cube'
        ),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()


res = 200
x_vec = np.linspace(-1, 1, res)
y_vec = np.linspace(-1, 1, res)
XI, YI = np.meshgrid(x_vec, y_vec)
x_lim = (-1, 1)
y_lim = (-1, 1)

v_theory = get_theoretical_volume(surface_A, surface_B, x_lim, y_lim)

ZA = surface_A(XI, YI)
ZB = surface_B(XI, YI)

V_computer = GridVolumeComputer(cp.asarray(XI.astype(np.float32)), cp.asarray(YI.astype(np.float32)), 
                        cp.asarray(ZA.astype(np.float32)), cp.asarray(ZB.astype(np.float32)), 
                        n_samples=2**13, qmc_type='sobol', debug_mode=True)

start = time.time()
est_res = V_computer.compute_volume()
est_volume, error_margin, error_percentage = est_res["est_volume"], est_res["error_margin"], est_res["error_percentage"]
print(f"精确理论体积: {v_theory:.6f}")
print(f"计算所得体积: {est_volume:.6f}")
print(f"统计波动范围: ±{error_margin:.6f}")
print(f"误差占总比: {error_percentage:.4f}%")
print(f"蒙特卡洛积分计算耗时: {time.time() - start:.4f}s")

visualize_volume_computation(V_computer, XI, YI, ZA, ZB)